In [1]:
import json
import sys
from pathlib import Path

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "textstyle").is_dir())
sys.path.insert(0, str(ROOT))
import matplotlib.pyplot as plt
import pandas as pd
import torch
from IPython.display import Image, display
from sklearn.metrics import auc, roc_curve

from textstyle.data import load_splits, read_corpus, verify_splits
from textstyle.evaluation import collect_logits, metrics_from_logits
from textstyle.models import build_model
from textstyle.predict import Predictor
from textstyle.settings import STYLES, TONES
from textstyle.tokenization import Collator, TextDataset, Vocabulary, tokenize
from textstyle.train import make_loader, train_experiment

torch.set_num_threads(2)
print("Python:", sys.version.split()[0], "PyTorch:", torch.__version__)
print("CUDA:", torch.cuda.is_available())

%matplotlib inline

Python: 3.12.10 PyTorch: 2.8.0+cpu
CUDA: False


## Очищення і захист від витоку даних

Початковий корпус містить назву literaly, яку нормалізуємо до literary. Текст Respond immediately. має дві різні стильові мітки. Виключаємо обидві копії та зберігаємо їх для ручного перегляду. Перевірка близьких триграм об'єднує схожі тексти в групи перед поділом.

Функції в textstyle/data.py створюють відтворюваний поділ, стратифікований за комбінацією style/tone. Один і той самий поділ використовують усі нейромережі. Пунктуацію та заперечення зберігаємо. Лематизація spaCy не є частиною нейронного pipeline.

In [2]:
cleaned, audit = read_corpus(ROOT / "data/raw/dataset.csv")
print(json.dumps({k: v for k, v in audit.items() if k != "excluded_conflicts"}, indent=2))
display(pd.DataFrame(audit["excluded_conflicts"]))
frames, manifest = load_splits(ROOT / "data/prepared")
verify_splits(frames)
display(
    pd.DataFrame(
        [{"split": name, "rows": len(frame), "groups": frame.group_id.nunique()} for name, frame in frames.items()]
    )
)
print("Перетин груп відсутній. Контрольні суми підтверджено.")

{
  "source_sha256": "078c8106329a2caffa6636ed02f4fcdbe23ab730693c051c1ecb6e7643200cb7",
  "source_rows": 1000,
  "empty_removed": 0,
  "duplicates_removed": 0,
  "conflicting_groups_removed": 1,
  "conflicting_rows_removed": 2,
  "label_aliases_fixed": 200,
  "clean_rows": 998
}


,text,style,tone
0,Respond immediately.,business,urgent
1,Respond immediately.,formal,urgent


,split,rows,groups
0,train,632,622
1,validation,158,155
2,test,208,191


Перетин груп відсутній. Контрольні суми підтверджено.


In [ ]:
vocabulary = Vocabulary.fit(frames["train"].text, max_length=256)
example = "PLEASE send the revised report today!"
print("Токени:", tokenize(example))
print("Ідентифікатори:", vocabulary.encode(example))
print("Розмір словника train:", len(vocabulary.tokens))
print("Токенізація не використовує validation або test для побудови словника.")

## Моделювання

**CNN:** навчувані embeddings, Conv1d з вікнами 3, 4, 5, маскований max pooling, dropout і дві голови.

**BiLSTM:** навчувані embeddings, двонапрямна LSTM, attention pooling і дві голови. Packed sequences прибирають вплив доповнення послідовностей.

**DistilBERT:** попередньо навчений англомовний encoder та дві нові голови. Донавчаємо ваги всіх шарів. Готовий токенізатор зберігається локально разом з вагами.

Функція втрат: L = CrossEntropy(style) + CrossEntropy(tone). Параметри оновлює AdamW. Використовуємо gradient clipping, зменшення learning rate та early stopping. Код циклу навчання розташований у textstyle/train.py.

In [ ]:
models = {
    "cnn": build_model("cnn", {"vocab_size": len(vocabulary.tokens)}),
    "bilstm": build_model("bilstm", {"vocab_size": len(vocabulary.tokens)}),
}
display(
    pd.DataFrame(
        [{"model": name, "parameters": sum(p.numel() for p in model.parameters())} for name, model in models.items()]
    )
)
batch = Collator()([TextDataset(frames["train"].head(2), vocabulary)[i] for i in range(2)])
style_logits, tone_logits = models["cnn"](batch["input_ids"], batch["lengths"])
loss = torch.nn.functional.cross_entropy(style_logits, batch["style_labels"]) + torch.nn.functional.cross_entropy(
    tone_logits, batch["tone_labels"]
)
loss.backward()
print("Дві голови:", style_logits.shape, tone_logits.shape, "Приклад loss:", round(loss.item(), 4))